# Fashion MNIST classification (Keras) with CNNs

## Libraries and constants

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"
import functools

import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset, random_split
from utils import plot_loss_acc_per_ep, seed_everything

In [2]:
# --- CONSTANTS ---
RANDOM_SEED = 42

# --- SEEDING FUNCTIONALITY ---
rng = seed_everything(RANDOM_SEED)

# for num_workers > 0 in PyTorch DataLoader, each worker will inherit the exact same random state from the main process
# This means every worker will generate the exact same "random" data augmentations. We fix this using worker_init_fn
torch_gen = torch.Generator()
torch_gen.manual_seed(RANDOM_SEED)

## Data import

In [3]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

### Data representation, data split

In [4]:
class FashionMNISTDataset(Dataset):
    def __init__(self, data: np.ndarray, targets: np.ndarray):
        self.data = torch.from_numpy(data).clone().float()      # pixels are floats to be able to normalize later
        self.targets = torch.from_numpy(targets).clone().int()  # class labels are integers (0-9)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

In [7]:
X_train_writable = X_train.copy()
y_train_writable = y_train.copy()

X_test_writable = X_test.copy()
y_test_writable = y_test.copy()

In [8]:
full_train_ds  = FashionMNISTDataset(X_train_writable, y_train_writable)
test_ds  = FashionMNISTDataset(X_test_writable, y_test_writable)

# Split train into train/validation (90/10)
train_size = int(0.9 * len(X_train))
val_size = len(full_train_ds) - train_size

train_ds, valid_ds = random_split(
    dataset=full_train_ds,
    lengths=[train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

In [18]:
print(f"Train: {len(train_ds)}, Validation: {len(valid_ds)}, Test: {len(test_ds)}")

Train: 54000, Validation: 6000, Test: 10000


## Creating and testing loaders

In [9]:
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True, generator=torch_gen)
valid_loader = DataLoader(valid_ds, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

In [10]:
# DataLoaders testing
images_batch, targets_batch = next(iter(train_loader))

print("Images batch shape :", images_batch.shape)
print("Targets batch shape:", targets_batch.shape)
print("Images data type   :", images_batch.dtype)
print("Targets data type  :", targets_batch.dtype)
print("Targets in batch   :", targets_batch)

Images batch shape : torch.Size([1024, 28, 28])
Targets batch shape: torch.Size([1024])
Images data type   : torch.float32
Targets data type  : torch.int32
Targets in batch   : tensor([9, 3, 3,  ..., 3, 6, 6], dtype=torch.int32)


## Data normalization

In [11]:
# Calculate the whole dataset's mean and std
n = 0; s = 0.0; ss = 0.0
for x, _ in train_loader:
    s  += x.sum().item()
    ss += (x**2).sum().item()
    n  += x.numel()

mean_ds = s / n
std_ds  = (ss/n - mean_ds**2) ** 0.5
variance_ds = std_ds ** 2

# Hardcode the stats into Keras
normalizer = keras.layers.Normalization(
    axis=None,
    mean=mean_ds,
    variance=variance_ds,
)

## Model architecting

Reference table:

| Year | Model                    | Total Params | FC-head Params | % in FC head | Typical block                                               | Conv layers | ImageNet acc (top 1) | ImageNet acc (top 5) |
| ---- | ------------------------ | ------------ | -------------- | ------------ | ----------------------------------------------------------- | ----------- | -------------------- | -------------------- |
| 1998 | LeNet-5                  | ~61,700      | ~59,100        | ~96%         | conv -> tanh -> avgpool                                     | 2           | ~99.0 (MNIST)        | —                    |
| 2012 | AlexNet                  | ~61,100,000  | ~58,600,000    | ~96%         | conv -> ReLU -> maxpool (LRN early)                         | 5           | 56.5                 | 79.1                 |
| 2014 | VGG-16                   | ~138,400,000 | ~123,600,000   | ~89%         | [conv3x3 -> ReLU] x2-3 -> maxpool                           | 13          | 71.6                 | 90.4                 |
| 2014 | VGG-19                   | ~143,700,000 | ~123,600,000   | ~86%         | [conv3x3 -> ReLU] x2-4 -> maxpool                           | 16          | 72.4                 | 90.9                 |
| 2015 | ResNet-18                | ~11,700,000  | ~513,000       | ~4%          | [conv3x3 -> BN -> ReLU -> conv3x3 -> BN] + skip -> ReLU     | 20          | 69.8                 | 89.1                 |
| 2015 | ResNet-34                | ~21,800,000  | ~513,000       | ~2%          | [conv3x3 -> BN -> ReLU -> conv3x3 -> BN] + skip -> ReLU     | 36          | 73.3                 | 91.4                 |

The first question is *what's the priority*. 

It is **accuracy**. 

*How many convolutions do I need* for best accuracy **at least**? Because we want the most efficient model.

Once we establish that, we will be assessing the model's performance empirically and adjust.

**Contemplation**: `Le-Net` is the simplest option, but its 2 layers might underperform, although it was specifically designed for datasets like this, namely MNIST. The next models by architectural simplicity are `AlexNet` with 5 convolutions, `VGG` (13-16 convolutions) and `ResNet` (20 convolutions). `ResNet` is the most modern among them. Its accuracy is on point. We are going to use it as a benchmark and build a teeny-tiny version of it. The typical block in the architecture is `[conv3x3 -> BN -> ReLU -> conv3x3 -> BN] + skip -> ReLU`. Given the complexity of the data, we won't need skip connections. The number of convolutions for such datasets shouldn't be too far from `Le-Net` then, say 4-10. If this under-/overfits empirically, we will be reducing them further. The typical block structure in `VGG` is `[conv3x3 -> ReLU] x2-3 -> maxpool`.

In [12]:
def cnn_block(n, activation_type="relu"):
    """Creates a sequential block consisting of Conv2D -> BatchNorm -> Activation."""
    return keras.Sequential([
        keras.layers.Conv2D(
            filters=n,
            kernel_size=(3, 3),
            strides=(1, 1),
            padding="valid",  # padding=0
            use_bias=False,     # no use in bias before BatchNorm
            activation=None,    # no use in activation before BatchNorm
        ),
        keras.layers.BatchNormalization(),
        keras.layers.Activation(activation_type),
    ])

def build_model(n, num_classes: int = 10, activation_type: str = "relu"):
    # 1. Define Input. Shape: (28,28,1) - grayscale
    inputs = keras.Input(shape=(28, 28, 1))

    # 2. Normalization Layer. Shape after: (28,28)
    x = keras.layers.Normalization()(inputs)

    # 3. First CNN block. Shape after: (n,26,26)
    x = cnn_block(n, activation_type)(x)

    # 4. Max Pooling. Shape after: (n,13,13)
    x = keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    # 5. Second CNN block. Shape after: (2n,11,11)
    x = cnn_block(2 * n, activation_type)(x)

    # 6. Third CNN block. Shape after: (4n,9,9)
    x = cnn_block(4 * n, activation_type)(x)

    # 7. Fourth CNN block. Shape after: (6n,7,7)
    x = cnn_block(6 * n, activation_type)(x)

    # 8. Flatten. Shape after: (1, 6nx7x7)
    x = keras.layers.Flatten()(x)

    # 9. Dense Classification Layer
    outputs = keras.layers.Dense(units=num_classes, activation="softmax")(x)

    # Construct the final model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [13]:
model = build_model(n=32, num_classes=10, activation_type="relu")
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_1 (Normalization) │ (None, 28, 28, 1)      │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 26, 26, 32)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 11, 11, 64)     │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 9, 9, 128)      │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_3 (Sequential)       │ (None, 7, 7, 192)      │       221,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        94,090 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 409,389 (1.56 MB)

 Trainable params: 408,554 (1.56 MB)

 Non-trainable params: 835 (3.26 KB)

The last fully-connected layer is `94,090` params for $n=32$. which is `23%` of all parameters, not critical given the reference table's.

This what I picked up on the web: "The creators of ResNet (and a slightly earlier network called Network In Network) realized flattening was a massive waste of parameters that caused severe overfitting. Instead of flattening, they introduced Global Average Pooling (GAP)."

In [16]:
def build_model(n, num_classes: int = 10, activation_type: str = "relu"):
    # 1. Define Input. Shape: (28,28,1) - grayscale
    inputs = keras.Input(shape=(28, 28, 1))

    # 2. Normalization Layer. Shape after: (28,28)
    x = keras.layers.Normalization()(inputs)

    # 3. First CNN block. Shape after: (n,26,26)
    x = cnn_block(n, activation_type)(x)

    # 4. Max Pooling. Shape after: (n,13,13)
    x = keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    # 5. Second CNN block. Shape after: (2n,11,11)
    x = cnn_block(2 * n, activation_type)(x)

    # 6. Third CNN block. Shape after: (4n,9,9)
    x = cnn_block(4 * n, activation_type)(x)

    # 7. Fourth CNN block. Shape after: (6n,7,7)
    x = cnn_block(6 * n, activation_type)(x)

    # 8. Average pooling. Shape after: (6n)
    x = keras.layers.GlobalAveragePooling2D()(x)

    # 9. Dense Classification Layer
    outputs = keras.layers.Dense(units=num_classes, activation="softmax")(x)

    # Construct the final model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [17]:
model = build_model(n=32, num_classes=10, activation_type="relu")
model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_2 (Normalization) │ (None, 28, 28, 1)      │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_4 (Sequential)       │ (None, 26, 26, 32)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_5 (Sequential)       │ (None, 11, 11, 64)     │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_6 (Sequential)       │ (None, 9, 9, 128)      │        74,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_7 (Sequential)       │ (None, 7, 7, 192)      │       221,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 192)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,930 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 317,229 (1.21 MB)

 Trainable params: 316,394 (1.21 MB)

 Non-trainable params: 835 (3.26 KB)

That's a massive parameter drop in the fully connected:

$\text{before: } 94,090 \quad \text{after: } 1,930$

## Training

In [34]:
def build_rand_search_model(hp):
    # Define the hyperparameters
    activation = hp.Choice("activation", ["relu", "leaky_relu", "gelu", "silu"])        # 4 options
    units = hp.Choice("units", values=[8, 16, 32, 64])                                  # 4 options

    mdl = build_model(n=units, activation_type=activation)

    lr = hp.Choice("lr", values=[0.0001, 0.0005, 0.001, 0.005])                 # 4 options
    optimizer_name = hp.Choice("optimizer", values=["adam", "sgd", "rmsprop"])  # 3 options

    if optimizer_name == "adam":
        optimizer_obj = keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == "sgd":
        optimizer_obj = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    elif optimizer_name == "rmsprop":
        optimizer_obj = keras.optimizers.RMSprop(learning_rate=lr)

    mdl.compile(
        optimizer=optimizer_obj,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return mdl

In [35]:
def build_loaders(batch_size: int, shuffle: bool, generator):
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle, generator=generator)
    val_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)

    return tr_loader, val_loader

def debug_tuner(tuner):
    trial_list = []

    # Loop through all trials stored in the tuner's oracle
    for trial_id, trial in tuner.oracle.trials.items():
        row = {
            "trial_id": trial_id,
            "score": trial.score,  # This will be your val_accuracy or val_loss
        }
        # Unpack and add the specific hyperparameters chosen for this trial
        row.update(trial.hyperparameters.values)
        trial_list.append(row)

    # Convert to DataFrame
    df_results = pd.DataFrame(trial_list)

    # Sort them so the best-performing models are at the top
    # (Change ascending=True if you are optimizing for loss instead of accuracy)
    df_results = df_results.sort_values(by="score", ascending=False)

    # Display the neat table!
    return df_results

In [37]:
keras.utils.set_random_seed(RANDOM_SEED)
torch_gen.manual_seed(RANDOM_SEED)
train_loader, valid_loader = build_loaders(batch_size=125, shuffle=True, generator=torch_gen)

tuner = kt.RandomSearch(
    build_rand_search_model,
    objective="val_accuracy",
    max_trials=15,
    directory="tuning",
    project_name="fashion_mnist",
    seed=RANDOM_SEED,                           # which HP combos get sampled
    overwrite=True,                             # start fresh every trial
)

tuner.search(
    x=train_loader,
    validation_data=valid_loader,
    epochs=10,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, verbose=1)],
    verbose=1,
    shuffle=False,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

Trial 15 Complete [00h 02m 25s]
val_accuracy: 0.737666666507721

Best val_accuracy So Far: 0.9226666688919067
Total elapsed time: 00h 43m 04s
{'activation': 'gelu', 'units': 64, 'lr': 0.0005, 'optimizer': 'adam'}


In [38]:
debug_tuner(tuner)

,trial_id,score,activation,units,lr,optimizer
13,13,0.922667,gelu,64,0.0005,adam
10,10,0.913833,relu,16,0.0010,adam
1,01,0.911000,silu,16,0.0010,adam
8,08,0.909500,silu,16,0.0005,rmsprop
3,03,0.895667,silu,8,0.0050,adam
12,12,0.888000,relu,8,0.0005,rmsprop
6,06,0.884000,relu,8,0.0010,rmsprop
2,02,0.880500,leaky_relu,8,0.0050,sgd
0,00,0.879500,gelu,8,0.0005,adam
11,11,0.878500,silu,8,0.0005,adam
